In [7]:
from googleapiclient.discovery import build
import pandas as pd
from datetime import datetime
import time
from pathlib import Path
import sys
parent_dir = Path.cwd().parent
sys.path.insert(0, str(parent_dir))
from config import YOUTUBE_API_KEY

# Initialize YouTube API client
youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)

# Test connection with a simple search
test_request = youtube.search().list(
    q="NASCAR 2025 highlights",
    part="snippet",
    maxResults=5,
    type="video"
)

test_response = test_request.execute()

print("YouTube API connected successfully!")
print(f"Found {len(test_response['items'])} videos")
print("\nSample results:")
for item in test_response['items']:
    print(f"  - {item['snippet']['title'][:60]}...")

YouTube API connected successfully!
Found 5 videos

Sample results:
  - Best finishes of the 2025 NASCAR Cup Series after a dramatic...
  - 2025 Daytona 500 Highlights | NASCAR Cup Series on FOX...
  - 2025 NASCAR Cup Series Full Race: YellaWood 500 at Talladega...
  - NASCAR Cup Series Highlights | 2026 Sonoma Raceway...
  - 🏁 LIVE NASCAR Cup Game Toyota/Save Mart 350 at Sonoma 2026 |...


In [24]:
#Create a list of all races in the 2025-6 NASCAR Cup Series seasons

#hardcode the path
race_results_df = pd.read_csv('~/Documents/nascar-visibility-analysis/code/data/processed/race_results_2025-6_clean.csv')

#year(1), race_number(2), race_name(3), Race_Date(4), track(5)
print(race_results_df.columns) #for debugging
races = race_results_df[['year', 'race_number', 'race_name', 'Race_Date', 'track']].drop_duplicates()
races = races.to_dict('records')  # Convert to list of dictionaries for easier iteration



print(f"Loaded {len(races)} races")


Index(['year', 'race_number', 'race_name', 'Race_Date', 'track', 'track_type',
       'track_miles', 'total_laps', 'caution_flags', 'caution_laps',
       'lead_changes', 'avg_speed_mph', 'pole_speed_mph', 'margin_of_victory',
       'attendance', 'Finish_Position', 'Start_Position', 'car_number',
       'Driver', 'Sponsor', 'Owner', 'car_make', 'Laps_Completed', 'status',
       'Laps_Led', 'Points', 'loop_avg_speed', 'loop_green_flag_passes',
       'loop_quality_passes', 'loop_fastest_lap', 'loop_driver_rating',
       'Primary_Sponsor', 'Positions_Gained'],
      dtype='str')
Loaded 53 races


In [25]:
def search_race_videos(youtube, race_name, year, track_name=None, max_results=10):
    """
    Search YouTube for videos related to a specific NASCAR race.

    Parameters:
    -----------
    youtube : googleapiclient.discovery.Resource
        Authenticated YouTube API client
    race_name : str
        Official race name
    track_name : str
        Track/location name (optional, improves search relevance)
    max_results : int
        Maximum videos to retrieve
    year : int
        Year of the race

    Returns:
    --------
    list : List of video IDs found
    """

    # Construct search query
    query = f"NASCAR {race_name} {year} highlights"
    if track_name:
        query = f"NASCAR {race_name} {track_name} {year}"

    try:
        search_response = youtube.search().list(
            q=query,
            part="snippet",
            maxResults=max_results,
            type="video",
            videoDuration="medium",  # Filter to medium length videos
            order="viewCount"  # Get most viewed first
        ).execute()

        videos = []
        for item in search_response['items']:
            videos.append({
                'video_id': item['id']['videoId'],
                'title': item['snippet']['title'],
                'channel': item['snippet']['channelTitle'],
                'publish_date': item['snippet']['publishedAt'],
                'description': item['snippet'].get('description', '')[:200]
            })

        return videos

    except Exception as e:
        print(f"  Error searching: {e}")
        return []

In [26]:
def get_video_statistics(youtube, video_ids):
    """
    Get detailed statistics for a list of videos.

    Parameters:
    -----------
    youtube : googleapiclient.discovery.Resource
        Authenticated YouTube API client
    video_ids : list
        List of YouTube video IDs

    Returns:
    --------
    dict : Dictionary mapping video_id to statistics
    """

    if not video_ids:
        return {}

    # YouTube API allows up to 50 videos per request
    stats = {}

    # Process in batches of 50
    for i in range(0, len(video_ids), 50):
        batch = video_ids[i:i+50]

        try:
            response = youtube.videos().list(
                part="statistics,contentDetails",
                id=",".join(batch)
            ).execute()

            for item in response['items']:
                video_id = item['id']
                statistics = item.get('statistics', {})

                stats[video_id] = {
                    'view_count': int(statistics.get('viewCount', 0)),
                    'like_count': int(statistics.get('likeCount', 0)),
                    'comment_count': int(statistics.get('commentCount', 0)),
                    'duration': item['contentDetails'].get('duration', '')
                }

        except Exception as e:
            print(f"  Error getting stats: {e}")

    return stats

In [ ]:
def collect_race_videos(youtube, races, videos_per_race=10):
    """
    Collect YouTube video data for all races in the season.

    Parameters:
    -----------
    youtube : googleapiclient.discovery.Resource
        Authenticated YouTube API client
    races : list
        List of race dictionaries with name and date
    videos_per_race : int
        Maximum videos to collect per race

    Returns:
    --------
    pd.DataFrame : Combined video data for all races
    """

    all_videos = []

    for i, race in enumerate(races, 1):
        race_name = race.get('Race_Name', race.get('name', ''))
        race_date = race.get('Race_Date', race.get('date', ''))
        race_num = race.get('Race_Number', race.get('race_number', i))

        print(f"\n[{i}/{len(races)}] Searching: {race_name}")

        # Search for race videos
        videos = search_race_videos(
            youtube,
            race_name,
            max_results=videos_per_race
        )

        if videos:
            # Get statistics for all videos
            video_ids = [v['video_id'] for v in videos]
            stats = get_video_statistics(youtube, video_ids)

            # Combine search results with statistics
            for video in videos:
                video_id = video['video_id']
                video_stats = stats.get(video_id, {})

                all_videos.append({
                    'race_number': race_num,
                    'race_name': race_name,
                    'race_date': race_date,
                    'video_id': video_id,
                    'video_title': video['title'],
                    'channel': video['channel'],
                    'publish_date': video['publish_date'],
                    'view_count': video_stats.get('view_count', 0),
                    'like_count': video_stats.get('like_count', 0),
                    'comment_count': video_stats.get('comment_count', 0),
                    'duration': video_stats.get('duration', ''),
                    'url': f"https://youtube.com/watch?v={video_id}"
                })

            print(f"  Found {len(videos)} videos, total views: {sum(v.get('view_count', 0) for v in all_videos if v['race_name'] == race_name):,}")
        else:
            print(f"  No videos found")

        # Rate limiting
        time.sleep(1)

    return pd.DataFrame(all_videos)

# Collect videos for all races
# Note: This uses API quota - monitor your usage
print("Starting YouTube video collection...")
print("=" * 50)

youtube_df = collect_race_videos(youtube, race_list, videos_per_race=10)

print("\n" + "=" * 50)
print(f"Collection complete!")
print(f"Total videos collected: {len(youtube_df)}")
print(f"Total views tracked: {youtube_df['view_count'].sum():,}")